In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 0 — CLAUDE CLIENT FROM COLAB SECRETS
# ============================================================
from anthropic import Anthropic

CLAUDE_API_KEY = ""

client = Anthropic(
    api_key=CLAUDE_API_KEY
)

print("Claude client initialized.")

Claude client initialized.


In [4]:

# ============================================================
# CELL 1 — CONFIG
# ============================================================

import os
import json
import re
import time
from collections import Counter

import pandas as pd

CV_RUN = 4   # change to 2,3,4,5 for other CV runs

DATA_PATH = "/content/drive/MyDrive/conversation_en.json"
SPLIT_DIR = "/content/drive/MyDrive/5class_5cv_frozen"

TRAIN_IDS_PATH = f"{SPLIT_DIR}/cv{CV_RUN}_train_ids.csv"
VAL_IDS_PATH   = f"{SPLIT_DIR}/cv{CV_RUN}_val_ids.csv"
TEST_IDS_PATH  = f"{SPLIT_DIR}/cv{CV_RUN}_test_ids.csv"

OUTPUT_DIR = f"/content/drive/MyDrive/5class_5cv_augmentation/cv{CV_RUN}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CORE_LABELS = ["Chitchat", "Revise"]
CORE_TARGETS = {
    "Chitchat": 100,
    "Revise": 100,
}

INQUIRE_TARGET = 100
MAX_INQUIRE_API_CALLS = 60

print("CV run:", CV_RUN)
print("Train IDs:", TRAIN_IDS_PATH)
print("Val IDs:", VAL_IDS_PATH)
print("Test IDs:", TEST_IDS_PATH)
print("Output dir:", OUTPUT_DIR)


CV run: 4
Train IDs: /content/drive/MyDrive/5class_5cv_frozen/cv4_train_ids.csv
Val IDs: /content/drive/MyDrive/5class_5cv_frozen/cv4_val_ids.csv
Test IDs: /content/drive/MyDrive/5class_5cv_frozen/cv4_test_ids.csv
Output dir: /content/drive/MyDrive/5class_5cv_augmentation/cv4


In [5]:

# ============================================================
# CELL 2 — LOAD FROZEN CV IDs
# ============================================================

train_cids = set(
    pd.read_csv(TRAIN_IDS_PATH)["conversation_id"].astype(str)
)

val_cids = set(
    pd.read_csv(VAL_IDS_PATH)["conversation_id"].astype(str)
)

test_cids = set(
    pd.read_csv(TEST_IDS_PATH)["conversation_id"].astype(str)
)

expected_train = 528 if CV_RUN == 1 else 529
expected_val = 132
expected_test = 166 if CV_RUN == 1 else 165

print("Train conversations:", len(train_cids))
print("Validation conversations:", len(val_cids))
print("Test conversations:", len(test_cids))

assert len(train_cids) == expected_train
assert len(val_cids) == expected_val
assert len(test_cids) == expected_test

assert train_cids.isdisjoint(val_cids)
assert train_cids.isdisjoint(test_cids)
assert val_cids.isdisjoint(test_cids)

assert len(train_cids | val_cids | test_cids) == 826

print("PASS: frozen CV split loaded and conversation-disjoint.")


Train conversations: 529
Validation conversations: 132
Test conversations: 165
PASS: frozen CV split loaded and conversation-disjoint.


In [6]:

# ============================================================
# CELL 3 — LOAD RAW DATA AND KEEP CV TRAIN CONVERSATIONS ONLY
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

cv_train_conversations = [
    conv
    for conv in raw_data
    if str(conv["conv_id"]) in train_cids
]

print(
    f"CV{CV_RUN} training conversations recovered:",
    len(cv_train_conversations)
)

assert len(cv_train_conversations) == expected_train

recovered_train_cids = {
    str(conv["conv_id"])
    for conv in cv_train_conversations
}

assert recovered_train_cids == train_cids

print(
    f"PASS: CV{CV_RUN} recovered exactly the "
    f"{expected_train} frozen training conversations."
)


CV4 training conversations recovered: 529
PASS: CV4 recovered exactly the 529 frozen training conversations.


In [7]:

# ============================================================
# CELL 4 — BUILD AUGMENTATION SOURCE POOL
# Chitchat + Revise + Inquire source turns from TRAIN only
# ============================================================

AUGMENT_LABELS = ["Chitchat", "Revise", "Inquire"]

augmentation_sources = []

for conv in cv_train_conversations:

    conv_id = str(conv["conv_id"])
    conversation = conv["conversation"]

    for i, msg in enumerate(conversation):

        if msg.get("role") != "user":
            continue

        intent = msg.get("intent")

        if intent not in AUGMENT_LABELS:
            continue

        previous_system_text = ""

        for previous_msg in reversed(conversation[:i]):
            if previous_msg.get("role") == "system":
                previous_system_text = previous_msg.get("content", "")
                break

        augmentation_sources.append({
            "conversation_id": conv_id,
            "turn_index": i,
            "system_text": previous_system_text,
            "user_text": msg.get("content", ""),
            "label": intent
        })

source_df = pd.DataFrame(augmentation_sources)

print(f"CV{CV_RUN} augmentation source distribution:")
print(source_df["label"].value_counts())

print("\nSource conversations by class:")
for label in AUGMENT_LABELS:
    subset = source_df[source_df["label"] == label]
    print(
        f"{label}: {len(subset)} source turns, "
        f"{subset['conversation_id'].nunique()} source conversations"
    )


CV4 augmentation source distribution:
label
Chitchat    177
Revise       40
Inquire      13
Name: count, dtype: int64

Source conversations by class:
Chitchat: 177 source turns, 139 source conversations
Revise: 40 source turns, 31 source conversations
Inquire: 13 source turns, 9 source conversations


In [8]:

# ============================================================
# CELL 5 — AUGMENTATION PROMPT
# Same basic prompt structure as 4-class experiment
# ============================================================

def build_augmentation_prompt(system_text, user_text, label):

    return f"""
You are an expert in e-commerce conversational AI and dialogue understanding.

Your task is to generate realistic, label-preserving variations of a conversation sample for intent classification training.

STRICT RULES:
- The intent label MUST remain EXACTLY the same.
- Do NOT change the user's goal or meaning.
- Do NOT introduce a different intent.
- Keep language natural, casual, and realistic.
- Do NOT make the text overly formal.
- Preserve conversational style, including short or imperfect phrasing when appropriate.
- If a variation changes the intent, do not generate it.
- If the Intent Label is "Inquire", the user must remain information-seeking or clarification-seeking.
- For Inquire, do NOT convert the user turn into a preference, requirement, acceptance, rejection, or revision.

INPUT
System: "{system_text}"
User: "{user_text}"
Intent Label: "{label}"

Generate exactly 3 variations:

Variation A - User-only
- Remove system message completely
- Rewrite only the user message
- Intent must still be clear

Variation B - Natural Conversation
- Slightly paraphrase the system message
- Slightly rewrite the user response
- Keep semantics identical

Variation C - Abstracted Context
- Replace system message with a short tag like [Clarify], [Recommend], [Confirm], or [Filter]
- Keep user response natural

Return ONLY raw valid JSON in this exact format.
Do not use markdown.
Do not use code fences.

[
  {{
    "type": "User-only",
    "input": "User: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Natural Conversation",
    "input": "System: <text>\\nUser: <text>",
    "label": "{label}"
  }},
  {{
    "type": "Abstracted Context",
    "input": "[Tag]\\nUser: <text>",
    "label": "{label}"
  }}
]
""".strip()


In [9]:

# ============================================================
# CELL 6 — OUTPUT CLEANING
# ============================================================

def clean_json_output(raw_output):

    text = raw_output.strip()

    if text.startswith("```json"):
        text = text[len("```json"):].strip()

    elif text.startswith("```"):
        text = text[len("```"):].strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text


def extract_user_text(input_text):

    if not isinstance(input_text, str):
        return ""

    match = re.search(
        r"User:\s*(.*)",
        input_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return input_text.strip()


In [10]:

# ============================================================
# CELL 8 — CLAUDE CALL
# ============================================================

def call_claude_for_variations(
    client,
    system_text,
    user_text,
    label
):

    prompt = build_augmentation_prompt(
        system_text,
        user_text,
        label
    )

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=600,
        temperature=0.4,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text
    cleaned = clean_json_output(raw_output)

    if not cleaned:
        raise ValueError("Empty Claude output")

    return json.loads(cleaned)


In [11]:

# ============================================================
# CELL 9 — GENERATE 100 CHITCHAT + 100 REVISE
# Same source-specific duplicate logic as 4-class CV script
# ============================================================

core_rows = []

for label in CORE_LABELS:

    label_sources = source_df[
        source_df["label"] == label
    ].reset_index(drop=True)

    target = CORE_TARGETS[label]
    seen = set()
    source_index = 0

    print(
        f"Starting {label}: target={target}, "
        f"source_turns={len(label_sources)}, "
        f"source_conversations={label_sources['conversation_id'].nunique()}"
    )

    while len([r for r in core_rows if r["intent"] == label]) < target:

        source_row = label_sources.iloc[
            source_index % len(label_sources)
        ]
        source_index += 1

        try:

            variations = call_claude_for_variations(
                client=client,
                system_text=source_row["system_text"],
                user_text=source_row["user_text"],
                label=label
            )

            if not isinstance(variations, list):
                continue

            for item in variations:

                current_count = len(
                    [r for r in core_rows if r["intent"] == label]
                )

                if current_count >= target:
                    break

                if item.get("label") != label:
                    continue

                new_user_text = extract_user_text(
                    item.get("input", "")
                )

                if not new_user_text:
                    continue

                normalized_new = new_user_text.strip().lower()
                normalized_original = (
                    str(source_row["user_text"]).strip().lower()
                )

                if normalized_new == normalized_original:
                    continue

                key = (
                    str(source_row["conversation_id"]),
                    int(source_row["turn_index"]),
                    normalized_new,
                    label
                )

                if key in seen:
                    continue

                seen.add(key)

                core_rows.append({
                    "conversation_id": str(source_row["conversation_id"]),
                    "turn_index": int(source_row["turn_index"]),
                    "role": "user",
                    "text": new_user_text,
                    "intent": label,
                    "is_augmented": True,
                    "augmentation_type": item.get("type", ""),
                    "source_user_text": source_row["user_text"],
                    "source_system_text": source_row["system_text"]
                })

            current_count = len(
                [r for r in core_rows if r["intent"] == label]
            )

            print(f"{label}: {current_count}/{target}")

            time.sleep(0.5)

        except Exception as e:
            print(
                f"Error | CV{CV_RUN} | {label} | "
                f"source {source_index}: {e}"
            )
            time.sleep(1)

core_df = pd.DataFrame(core_rows)

print("\nCore synthetic distribution:")
print(core_df["intent"].value_counts())

assert core_df["intent"].value_counts().get("Chitchat", 0) == 100
assert core_df["intent"].value_counts().get("Revise", 0) == 100
assert len(core_df) == 200

print("PASS: 100 Chitchat + 100 Revise generated.")


Starting Chitchat: target=100, source_turns=177, source_conversations=139
Chitchat: 2/100
Chitchat: 4/100
Chitchat: 6/100
Chitchat: 8/100
Chitchat: 11/100
Chitchat: 13/100
Chitchat: 16/100
Chitchat: 19/100
Chitchat: 22/100
Chitchat: 25/100
Chitchat: 28/100
Chitchat: 31/100
Chitchat: 34/100
Chitchat: 37/100
Chitchat: 40/100
Chitchat: 43/100
Chitchat: 46/100
Chitchat: 49/100
Chitchat: 51/100
Chitchat: 53/100
Chitchat: 56/100
Chitchat: 59/100
Chitchat: 61/100
Chitchat: 63/100
Chitchat: 66/100
Chitchat: 69/100
Chitchat: 72/100
Chitchat: 75/100
Chitchat: 78/100
Chitchat: 80/100
Chitchat: 82/100
Chitchat: 85/100
Chitchat: 88/100
Chitchat: 90/100
Chitchat: 90/100
Chitchat: 92/100
Chitchat: 94/100
Chitchat: 96/100
Chitchat: 98/100
Chitchat: 100/100
Starting Revise: target=100, source_turns=40, source_conversations=31
Revise: 3/100
Revise: 6/100
Revise: 9/100
Revise: 12/100
Revise: 14/100
Revise: 16/100
Revise: 19/100
Revise: 22/100
Revise: 25/100
Revise: 27/100
Revise: 30/100
Revise: 33/100
Re

In [12]:

# ============================================================
# CELL 10 — VERIFY + SAVE CORE AUGMENTATION
# ============================================================

core_cids = set(core_df["conversation_id"].astype(str))

assert core_cids.issubset(train_cids)
assert core_cids.isdisjoint(val_cids)
assert core_cids.isdisjoint(test_cids)

source_keys = set(
    zip(
        source_df["conversation_id"].astype(str),
        source_df["turn_index"].astype(int),
        source_df["label"].astype(str),
    )
)

core_keys = set(
    zip(
        core_df["conversation_id"].astype(str),
        core_df["turn_index"].astype(int),
        core_df["intent"].astype(str),
    )
)

assert core_keys.issubset(source_keys)

CORE_OUTPUT_PATH = (
    f"{OUTPUT_DIR}/"
    f"cv{CV_RUN}_synthetic_100_chitchat_100_revise.csv"
)

core_df.to_csv(
    CORE_OUTPUT_PATH,
    index=False
)

print("Synthetic ∩ Validation:", len(core_cids & val_cids))
print("Synthetic ∩ Test:", len(core_cids & test_cids))
print("PASS: core augmentation is train-only and provenance-safe.")
print("Saved core augmentation:", CORE_OUTPUT_PATH)


Synthetic ∩ Validation: 0
Synthetic ∩ Test: 0
PASS: core augmentation is train-only and provenance-safe.
Saved core augmentation: /content/drive/MyDrive/5class_5cv_augmentation/cv4/cv4_synthetic_100_chitchat_100_revise.csv


In [13]:
# ============================================================
# CELL 11 — OPTIMIZED INQUIRE GENERATION
# ONE API CALL PER ORIGINAL INQUIRE SOURCE TURN
# ============================================================

inquire_sources = (
    source_df[source_df["label"] == "Inquire"]
    .reset_index(drop=True)
)

print("Inquire source turns:", len(inquire_sources))
print(
    "Inquire source conversations:",
    inquire_sources["conversation_id"].nunique()
)

assert len(inquire_sources) > 0

# 3 requested variations per original source turn
INQUIRE_TARGET = len(inquire_sources) * 3

inquire_rows = []
seen_inquire = set()

api_calls = 0

for source_index, source_row in inquire_sources.iterrows():

    api_calls += 1

    try:

        variations = call_claude_for_variations(
            client=client,
            system_text=source_row["system_text"],
            user_text=source_row["user_text"],
            label="Inquire"
        )

        if not isinstance(variations, list):
            continue

        for item in variations:

            if item.get("label") != "Inquire":
                continue

            new_user_text = extract_user_text(
                item.get("input", "")
            )

            if not new_user_text:
                continue

            normalized_new = (
                new_user_text
                .strip()
                .lower()
            )

            normalized_original = (
                str(source_row["user_text"])
                .strip()
                .lower()
            )

            # Reject unchanged source copy
            if normalized_new == normalized_original:
                continue

            # Same source-specific duplicate logic
            # as original 4-class augmentation
            key = (
                str(source_row["conversation_id"]),
                int(source_row["turn_index"]),
                normalized_new,
                "Inquire"
            )

            if key in seen_inquire:
                continue

            seen_inquire.add(key)

            inquire_rows.append({
                "conversation_id":
                    str(source_row["conversation_id"]),

                "turn_index":
                    int(source_row["turn_index"]),

                "role": "user",

                "text": new_user_text,

                "intent": "Inquire",

                "is_augmented": True,

                "augmentation_type":
                    item.get("type", ""),

                "source_user_text":
                    source_row["user_text"],

                "source_system_text":
                    source_row["system_text"]
            })

        print(
            f"Source {source_index + 1}/"
            f"{len(inquire_sources)} | "
            f"accepted={len(inquire_rows)}"
        )

        time.sleep(0.5)

    except Exception as e:

        print(
            f"Error | CV{CV_RUN} | "
            f"Inquire source {source_index + 1}: {e}"
        )

        time.sleep(1)


inquire_df = pd.DataFrame(inquire_rows)

print("\n==============================")
print("INQUIRE GENERATION SUMMARY")
print("==============================")
print("Source turns:", len(inquire_sources))
print("API calls used:", api_calls)
print("Maximum candidates:", INQUIRE_TARGET)
print("Accepted synthetic Inquire:", len(inquire_df))

Inquire source turns: 13
Inquire source conversations: 9
Source 1/13 | accepted=1
Source 2/13 | accepted=4
Source 3/13 | accepted=6
Source 4/13 | accepted=9
Source 5/13 | accepted=11
Source 6/13 | accepted=14
Source 7/13 | accepted=16
Source 8/13 | accepted=19
Source 9/13 | accepted=21
Source 10/13 | accepted=24
Source 11/13 | accepted=27
Source 12/13 | accepted=29
Source 13/13 | accepted=32

INQUIRE GENERATION SUMMARY
Source turns: 13
API calls used: 13
Maximum candidates: 39
Accepted synthetic Inquire: 32


In [14]:

# ============================================================
# CELL 12 — VERIFY + SAVE INQUIRE AUGMENTATION
# ============================================================

if len(inquire_df) > 0:

    inquire_cids = set(
        inquire_df["conversation_id"].astype(str)
    )

    assert inquire_cids.issubset(train_cids)
    assert inquire_cids.isdisjoint(val_cids)
    assert inquire_cids.isdisjoint(test_cids)

    inquire_keys = set(
        zip(
            inquire_df["conversation_id"].astype(str),
            inquire_df["turn_index"].astype(int),
            inquire_df["intent"].astype(str),
        )
    )

    assert inquire_keys.issubset(source_keys)

    INQUIRE_OUTPUT_PATH = (
        f"{OUTPUT_DIR}/"
        f"cv{CV_RUN}_synthetic_inquire_{len(inquire_df)}.csv"
    )

    inquire_df.to_csv(
        INQUIRE_OUTPUT_PATH,
        index=False
    )

    print("Synthetic Inquire ∩ Validation:", len(inquire_cids & val_cids))
    print("Synthetic Inquire ∩ Test:", len(inquire_cids & test_cids))
    print("PASS: Inquire augmentation is train-only and provenance-safe.")
    print("Saved Inquire augmentation:", INQUIRE_OUTPUT_PATH)

else:
    print("No Inquire rows generated; nothing saved.")


Synthetic Inquire ∩ Validation: 0
Synthetic Inquire ∩ Test: 0
PASS: Inquire augmentation is train-only and provenance-safe.
Saved Inquire augmentation: /content/drive/MyDrive/5class_5cv_augmentation/cv4/cv4_synthetic_inquire_32.csv


In [15]:

# ============================================================
# CELL 13 — COMBINE + FINAL AUDIT + SAVE
# ============================================================

combined_df = pd.concat(
    [core_df, inquire_df],
    ignore_index=True
)

print("Final combined distribution:")
print(combined_df["intent"].value_counts())

print("\nTotal synthetic:", len(combined_df))

combined_cids = set(
    combined_df["conversation_id"].astype(str)
)

assert combined_cids.issubset(train_cids)
assert combined_cids.isdisjoint(val_cids)
assert combined_cids.isdisjoint(test_cids)

combined_keys = set(
    zip(
        combined_df["conversation_id"].astype(str),
        combined_df["turn_index"].astype(int),
        combined_df["intent"].astype(str),
    )
)

assert combined_keys.issubset(source_keys)

# Post-generation duplicate audit
normalized = (
    combined_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

global_exact_duplicates = int(normalized.duplicated().sum())

print("Global exact duplicate generated texts:", global_exact_duplicates)
print("Synthetic ∩ Validation:", len(combined_cids & val_cids))
print("Synthetic ∩ Test:", len(combined_cids & test_cids))

COMBINED_OUTPUT_PATH = (
    f"{OUTPUT_DIR}/"
    f"cv{CV_RUN}_synthetic_100_chitchat_100_revise_"
    f"{len(inquire_df)}_inquire.csv"
)

combined_df.to_csv(
    COMBINED_OUTPUT_PATH,
    index=False
)

print("\nPASS: final combined augmentation is train-only and provenance-safe.")
print("Saved combined augmentation:", COMBINED_OUTPUT_PATH)


Final combined distribution:
intent
Chitchat    100
Revise      100
Inquire      32
Name: count, dtype: int64

Total synthetic: 232
Global exact duplicate generated texts: 52
Synthetic ∩ Validation: 0
Synthetic ∩ Test: 0

PASS: final combined augmentation is train-only and provenance-safe.
Saved combined augmentation: /content/drive/MyDrive/5class_5cv_augmentation/cv4/cv4_synthetic_100_chitchat_100_revise_32_inquire.csv
